# Generate a Favorites PDF

Generate a PDF of all the favorite images across a set of Agouti projects. Great for sharing with collaborators.

Update the following variables with your Agouti credentials and project IDs, then run the notebook to generate the PDF.

In [ ]:
AGOUTI_EMAIL = ""
AGOUTI_PASSWORD = ""
AGOUTI_PROJECT_IDS = [
    "11111111-1111-1111-1111-111111111111",
]



Install the agoutix package if you haven't already:


In [ ]:
%pip install agoutix

Authenticate

In [ ]:
from agoutix import Agouti

agouti = Agouti(AGOUTI_EMAIL, AGOUTI_PASSWORD)

Build the PDF!

In [ ]:
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

all_images = []
all_images_low_res = []

for project_id in tqdm(AGOUTI_PROJECT_IDS):
    favorites = {}

    for i in range(10):
        # Agouti returns 3 random favorites, so we loop a few times to try to get more unique ones
        new_favs = agouti.list_favorites(project_id)
        for fav in new_favs:
            favorites[fav.id] = fav

    favorites = list(favorites.values())
    favorites.sort(key=lambda x: x.id, reverse=True)

    for asset in favorites:
        image_raw, _ = agouti.get_asset_file(asset.id)
        image = Image.open(BytesIO(image_raw)).convert("RGB")
        image_low_res = image.copy().resize((image.width // 6, image.height // 6))
        all_images.append(image)
        all_images_low_res.append(image_low_res)

if all_images:
    all_images[0].save(
        "favorites.pdf",
        "PDF",
        resolution=100.0,
        save_all=True,
        append_images=all_images[1:],
    )
    all_images_low_res[0].save(
        "favorites_low_res.pdf",
        "PDF",
        resolution=100.0,
        save_all=True,
        append_images=all_images_low_res[1:],
    )